# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, a clutch term, and consistency. A career rate is the decay-weighted mean of the rows that player has. A season with no rows is left out of both the total and the weight. The weights are 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or an other-type extra. Super overs (innings 3 and 4) are kept; every super-over ball is a death ball. An other-type extra is re-bowled like a no-ball. Every extra carries its run amount, so a team total includes wides, no-balls, byes, and leg-byes. Every run off a wide is charged to the bowler, and so is the one run of a no-ball. Byes run off a no-ball are not. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied. The second super over chases the first super over's total plus one.

Pressure is the occasion times the competition times the ball, plus a shock from the last six balls and a break from the previous eighteen. The current ball is not included. The shock starts with dots and wickets for a batter, and fours and sixes for a bowler, then adds the wickets above xW and the charged runs above xR on those same six balls. Bowling bands are cut inside pace and inside spin, so the new ball does not put more pace bowlers into the high band. A recognized batter is anyone at positions 1 to 7. Zero, medium, and high are the bottom half, the next 30 percent, and the top fifth of that pressure.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal", len(legal), "of which super overs", int(legal["is_super_over"].sum()))
print("five-season weight total", round(ratings.W_SUM, 4))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))

deliveries 200999 legal 192445 of which super overs 33
five-season weight total 4.0951


,balls,share
band,,
zero,96219,50.0
medium,57485,29.9
high,38741,20.1


## Elo

Team strength updates after each match. Batting strength and bowling strength update after each legal ball, on three ladders: IPL, T20I, and lower T20. Emerging Teams and Rising Stars are the lower ladder. A rating learned there does not carry into a T20I, and a ball there counts as half a ball in the career rates and the ranking floor. A player with few balls is held near 1500, and that hold loosens as the career lengthens. Venue strength uses only balls already bowled at that ground. Expected runs and the expected wicket chance are also taken from balls already bowled, split by phase, pace or spin, and ladder, and pulled toward the coarser rate with 30 balls. Before any ball, the starting assumption is 1.0 run and a 0.05 wicket chance. A batter is priced on bat runs. A bowler is priced on charged runs, which are 0 on a bye, leg-bye, or penalty, and the bowling rating uses that same charged total.

In [2]:
charged_gap = legal["bat_runs"] - legal["charged_runs"]
print("balls where bat runs differ from charged runs", int(charged_gap.ne(0).sum()))
print("sum of bat runs minus charged runs", round(float(charged_gap.sum()), 3))
legal = ratings.fit_elo(legal)
summary = legal.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))

balls where bat runs differ from charged runs 1
sum of bat runs minus charged runs 1.0


k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 931.5
batter_elo_max: 2571.6
batter_elo_sd: 141.8
bowler_elo_min: 1059.8
bowler_elo_max: 2136.6
bowler_elo_sd: 118.0
team_elo_min: 1361.5
team_elo_max: 1647.1
grounds: 97
step_sizes: as written


## Batter scores

Baseline is strike rate and average against the phases that batter actually plays. The three bands are the strength-adjusted ball value inside zero, medium, and high pressure. Peers compare an innings with teammates and the opposition, and the opposition gap is capped at 3, the same cap as the teammate gap. Clutch uses a high-band spell against the player's own rate. A spell below that rate stays negative. A spell above it is scaled by how far the player also beat their teammates, and a negative teammate gap does not add a bonus. Consistency prefers a narrower spread of match values.

Each term is a z-score inside the batting role, so a top-order batter is compared with other top-order batters, a middle-order batter with the middle order, and a lower-order batter with the lower order. Inside each role the reference is still players with at least 30 decay-weighted legal balls, and each z-score stops at 3. The composite is 0.20 baseline, 0.10 zero, 0.15 medium, 0.20 high, 0.15 clutch, 0.10 peers, and 0.10 consistency. The seven weights sum to 1.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 216


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Rilee Rossouw,top order,1.413,1.389,2.009,0.389,3.000,0.008,2.059,0.691,717,172.771,32.591,302.519,55.574
1,2.0,Tim David,middle order,1.171,2.359,1.774,1.786,0.419,-0.427,3.000,-0.663,2142,177.025,34.548,998.270,313.857
2,3.0,Mahendra Singh Dhoni,lower order,1.119,1.789,-0.745,1.498,1.037,3.000,1.586,-2.051,684,149.672,31.091,350.221,127.448
3,4.0,George Linde,lower order,1.059,1.091,-0.381,1.972,2.028,0.262,1.732,-0.354,227,162.143,18.917,130.740,61.550
4,5.0,Ashutosh Sharma,lower order,1.028,2.097,0.685,1.313,1.617,-0.141,0.974,-0.563,570,168.460,27.488,300.350,93.400
5,6.0,Tom Banton,middle order,0.954,1.207,1.818,0.188,2.233,-0.073,1.418,-0.746,436,159.707,31.143,238.776,68.642
6,7.0,Vaibhav Sooryavanshi,top order,0.914,2.458,2.969,3.000,0.003,-2.171,3.000,-3.000,1505,222.035,41.985,607.000,41.650
7,8.0,Donovan Ferreira,middle order,0.913,0.973,0.724,1.915,0.969,2.013,1.047,-2.421,561,164.516,24.391,316.689,85.793
8,9.0,Muhammad Waseem,top order,0.903,0.893,-0.930,2.372,1.138,0.008,1.673,0.653,669,145.752,37.167,388.672,43.236
9,10.0,Bevon Jacobs,middle order,0.901,1.156,0.584,1.078,2.234,-0.073,0.430,-0.294,275,146.277,39.286,185.900,30.600


roles in the batter top 10
role
middle order    4
top order       3
lower order     3


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's pace or spin and the phases they bowl. A lower rate is a higher score. The high band, clutch, and peers use the same direction.

In [4]:
bowlers = ratings.score_bowlers(legal, work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 226


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Corbin Bosch,Pace,1.335,1.199,1.442,-0.556,2.637,3.000,0.585,-0.018,33.0,8.632,11.121,346.000,83.700
1,2.0,Matthew Humphreys,Spin,1.324,1.166,1.324,0.127,1.468,2.679,1.713,0.723,20.0,7.141,15.250,277.028,31.330
2,3.0,Haider Ali,Spin,1.076,1.233,0.035,0.747,1.876,0.254,2.152,0.857,15.0,6.422,18.000,247.800,58.500
3,4.0,Mujeeb Ur Rahman,Spin,1.041,1.253,-0.553,0.412,1.460,1.728,1.691,0.639,48.0,7.032,17.083,675.993,117.913
4,5.0,Imad Wasim,Spin,1.025,1.446,1.156,0.318,1.543,0.096,3.000,-0.500,15.0,6.146,16.400,188.568,38.475
5,6.0,Dwayne Bravo,Pace,0.930,0.850,0.157,1.554,1.227,1.090,-0.530,1.555,16.0,8.709,12.875,135.157,39.366
6,7.0,Sunil Narine,Spin,0.892,0.565,1.289,-0.573,1.001,0.698,3.000,1.315,66.0,6.865,23.227,1249.702,260.169
7,8.0,Jasprit Bumrah,Pace,0.890,1.285,0.511,1.447,0.360,0.004,2.940,-0.022,85.0,7.268,20.094,1454.183,303.882
8,9.0,Shoriful Islam,Pace,0.854,0.583,-0.905,0.996,1.735,1.987,-0.292,0.628,41.0,8.309,19.268,649.565,87.655
9,10.0,Haris Rauf,Pace,0.749,1.025,0.051,0.082,0.298,3.000,-0.170,0.339,76.0,8.670,13.513,798.394,169.569


roles in the bowler top 10
role
Pace    5
Spin    5


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 30 of those in the high band. Anyone under the line stays in the full table with no rank. The two lists are saved under `ratings/`.

In [5]:
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: career rate is the decay-weighted mean of rows played; the five weights sum to", round(ratings.W_SUM, 4))
print("decision: elo step sizes", summary["step_sizes"])

wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: career rate is the decay-weighted mean of rows played; the five weights sum to 4.0951
decision: elo step sizes as written


## Deck exports

These files are the evidence behind each stage: cleaning counts, pressure distributions, Elo bounds, calibration, component weights, and the rankings. They are written from the frames above.

In [6]:
notes = ratings.write_deck_exports(balls, work, legal, batters, bowlers)
print("wrote exports/")
for item, value in notes.items():
    print(item + ":", value)

wrote exports/
sample_match_id: 40767
sample_innings: 2
sample_date: 2025-06-03
sample_stage: Final
sample_competition: IPL 2025
sample_home: Royal Challengers Bengaluru
sample_away: Punjab Kings
trajectory_player: Jos Buttler
trajectory_up: 45394
trajectory_down: 20206
trajectory_up_date: 2026-07-11
trajectory_down_date: 2023-04-19
trajectory_up_runs: 131.0
trajectory_down_runs: 36.0
